### RAG with Tabalar Data


In [19]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('Chatbot_rag_v1') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [62]:
# !pip install langchain
# !pip install langchain_community
# !pip install -qU langchain-ollama
# !pip install python-dotenv
# !pip install -qU langchain-qdrant

In [20]:
import os
from dotenv import load_dotenv

from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import ChatOllama


from pyspark.sql.types import StructType
from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, col

In [21]:
%run ./01_Config_env.ipynb

In [22]:
load_dotenv('./.env')
OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
token = os.getenv("API_TOKEN")

In [23]:
 %run ./02_Common.ipynb

In [24]:
%run ./03_Get_data.ipynb

Dados disponiveis:
root
 |-- c: string (nullable = true)
 |-- cl: string (nullable = true)
 |-- sl: string (nullable = true)
 |-- lt0: string (nullable = true)
 |-- lt1: string (nullable = true)
 |-- qv: string (nullable = true)
 |-- vs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- p: string (nullable = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- ta: string (nullable = true)
 |    |    |-- py: string (nullable = true)
 |    |    |-- px: string (nullable = true)
 |    |    |-- sv: string (nullable = true)
 |    |    |-- is: string (nullable = true)



### Visualizar e pegar uma amostra dos dados

In [25]:
df_posicao.show(10)

+-------+-----+---+--------------------+--------------------+---+--------------------+
|      c|   cl| sl|                 lt0|                 lt1| qv|                  vs|
+-------+-----+---+--------------------+--------------------+---+--------------------+
|707A-10|34758|  2|METRÔ PÇA. DA ÁRVORE|          JD. ÂNGELA|  9|[{73968, true, 20...|
|2590-10|33680|  2|     PQ. D. PEDRO II|   UNIÃO DE VL. NOVA|  5|[{56327, true, 20...|
|2726-10|  940|  1|         METRÔ PENHA|            LIMOEIRO|  7|[{35990, true, 20...|
|208V-10|32975|  2|TERM. PQ. D. PEDR...|TERM. A. E. CARVALHO| 20|[{31150, true, 20...|
|8065-10|33189|  2|                LAPA|      HAB. TURÍSTICA|  2|[{10609, true, 20...|
|8050-10|33180|  2|                LAPA|      PQ. MORRO DOCE|  4|[{11034, true, 20...|
|978A-10| 1369|  1|   METRÔ BARRA FUNDA|  TERM. CACHOEIRINHA|  3|[{12478, true, 20...|
|1732-10|  671|  1| TERM. AMARAL GURGEL|         VL. SABRINA|  4|[{21755, true, 20...|
|263J-10|32987|  2|CONJ. JOSÉ BONIFÁCIO|   

In [26]:
from pyspark.sql.functions import explode, col

df = df_posicao.select(
    col('c').alias('Letreiro_Linha'),
    col('cl').alias('Linha'),
    col('sl').alias('Sentido'),
    col('lt0').alias('Destino_Linha'),
    col('lt1').alias('Origem_Linha'),
    col('qv').cast('int').alias('Quantidade_Veiculos')
    
).limit(10)

df.createOrReplaceTempView("tbl_bus_posicao")

In [27]:
spark.sql("Select * from tbl_bus_posicao").show()

+--------------+-----+-------+--------------------+--------------------+-------------------+
|Letreiro_Linha|Linha|Sentido|       Destino_Linha|        Origem_Linha|Quantidade_Veiculos|
+--------------+-----+-------+--------------------+--------------------+-------------------+
|       707A-10|34758|      2|METRÔ PÇA. DA ÁRVORE|          JD. ÂNGELA|                  9|
|       2590-10|33680|      2|     PQ. D. PEDRO II|   UNIÃO DE VL. NOVA|                  5|
|       2726-10|  940|      1|         METRÔ PENHA|            LIMOEIRO|                  7|
|       208V-10|32975|      2|TERM. PQ. D. PEDR...|TERM. A. E. CARVALHO|                 20|
|       8065-10|33189|      2|                LAPA|      HAB. TURÍSTICA|                  2|
|       8050-10|33180|      2|                LAPA|      PQ. MORRO DOCE|                  4|
|       978A-10| 1369|      1|   METRÔ BARRA FUNDA|  TERM. CACHOEIRINHA|                  3|
|       1732-10|  671|      1| TERM. AMARAL GURGEL|         VL. SABRIN

## Funções Auxiliares

In [28]:
# Coletar metadado da tabela
def describe_table(df, table_name="tbl_bus_posicao"):
    colunas = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{colunas}"

# Remover formatação do código gerado
import re

def limpar_sql(resposta_modelo):
    # Remove blocos de código markdown e espaços extras
    sql = re.sub(r"```sql|```", "", resposta_modelo, flags=re.IGNORECASE).strip()
    return sql


### Carrega modelo (Mistral 7B)

In [35]:
llm = ChatOllama(
    model="mistral:latest", 
    base_url=OLLAMA_API_URL,
    temperature=0.3,
    num_predict=200,
    top_k=30,
    top_p=0.9,
    repeat_penalty=1.1
    
) 

### Configurar Promps (roles System, Human)

In [30]:
# Prompt para gerar SQL (com roles)
prompt_sql = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um especialista em dados. Gere apenas a consulta SQL."),
    HumanMessagePromptTemplate.from_template(
        "Com base na estrutura da tabela abaixo:\n\n{schema}\n\n"
        "Escreva uma consulta SQL (somente a SQL) para responder:\n{pergunta}"
    )
])


In [31]:
# Prompt para gerar resposta para o usuário (com roles)
prompt_resposta = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um assistente de dados."),
    HumanMessagePromptTemplate.from_template(
        "Pergunta: {pergunta}\n\nResultado da consulta:\n{resultado}\n\n"
        "Gere uma resposta clara e amigável para o usuário."
    )
])

### Chatbot com RAG

In [36]:
def augmented_response(pergunta):
    print(f"Pergunta: {pergunta}")

    #Gerar SQL com role, a partir da pergunta do usuário (Especialista de Dados)
    schema_txt = describe_table(df, "tbl_bus_posicao")
    sql_chain = prompt_sql | llm
    sql_result = sql_chain.invoke({"pergunta": pergunta, "schema": schema_txt})
    sql_query = limpar_sql(sql_result.content)
    print(f"\n🤖💡 SQL Gerado:\n{sql_query}")

    # Executar SQL gerado pela "role especialista de dados" no Spark
    try:
        resultado_df = spark.sql(sql_query).toPandas().to_dict(orient="records")
    except Exception as e:
        print(f"❌ Erro na execução da SQL: {e}")
        return

    # Gerar resposta final com role Human (Assistente de Dados)
    resposta_chain = prompt_resposta | llm
    resposta_result = resposta_chain.invoke({
        "pergunta": pergunta,
        "resultado": resultado_df
    })
    resposta = resposta_result.content.strip()

    print(f"\n🤖 Resposta:\n{resposta}")

In [37]:
augmented_response("Qual o total de viculos?")

Pergunta: Qual o total de viculos?

🤖💡 SQL Gerado:
SELECT SUM(Quantidade_Veiculos) AS Total_de_Veiculos FROM tbl_bus_posicao;

🤖 Resposta:
O número total de veículos é 61.
